### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [22]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

# model=init_chat_model(model='google_genai:gemini-2.5-flash-lite')
model=init_chat_model(model='groq:qwen/qwen3-32b')
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x117bdc550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x117bddf90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str=Field(description="The Title of the movie")
    year:str=Field(description="The year the movie released")
    director:str=Field(description="The Movie's director name")
    collection:float=Field(description="The amout tha movie collect on box office")
    rating:float=Field(description="The movie rating out of 4")

In [5]:
model_with_struc = model.with_structured_output(Movie)

In [6]:
res = model.invoke("Provide the details about the movie Interstellar")
print(res.content)

**Interstellar** is a 2014 American science fiction film directed by Christopher Nolan. It's a visually stunning and intellectually stimulating epic that blends scientific concepts with a deeply emotional human story.

Here's a breakdown of its key details:

**Logline:**

A team of astronauts travels through a wormhole in space in an attempt to ensure humanity's survival.

**Plot Summary (Spoiler-lite):**

In a near-future Earth ravaged by blight and dust storms, humanity faces extinction. Crops are failing, and breathable air is becoming scarce. Cooper, a former NASA pilot turned farmer, is struggling to provide for his family. His life takes a dramatic turn when he discovers a secret NASA facility dedicated to a last-ditch effort: finding a new habitable planet for humanity. He's recruited for a mission through a newly discovered wormhole near Saturn, which offers the only hope of finding such a planet. The mission is fraught with danger, complex scientific challenges, and the immens

In [11]:
res = model_with_struc.invoke("Provide the details about the movie Avanges")
res

Movie(title='The Avengers', year='2012', director='Joss Whedon', collection=1518812988.0, rating=3.8)

### Message output alongside parsed structure

In [12]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str=Field(description="The Title of the movie")
    year:str=Field(description="The year the movie released")
    director:str=Field(description="The Movie's director name")
    collection:float=Field(description="The amout tha movie collect on box office")
    rating:float=Field(description="The movie rating out of 10")

model_with_struc = model.with_structured_output(Movie, include_raw=True)
res = model_with_struc.invoke("Provide the details about the movie Moana")
res

{'raw': AIMessage(content='{\n  "title": "Moana",\n  "year": "2016",\n  "director": "Ron Clements, John Musker",\n  "collection": 687.2,\n  "rating": 7.6\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f46df-1543-7003-8a19-fe44c360844c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 57, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Moana', year='2016', director='Ron Clements, John Musker', collection=687.2, rating=7.6),
 'parsing_error': None}

### Nested Structure

In [15]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str

class MovieData(BaseModel):
    title:str=Field(description="The Title of the movie")
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(None, description="Budget in Crore in INR")

model_with_struc_2 = model.with_structured_output(MovieData)

res2 = model_with_struc_2.invoke("rovide the details about the movie Avengers: Endgame")
res2

MovieData(title='Avengers: Endgame', cast=[Actor(name='Robert Downey Jr.', role='Iron Man'), Actor(name='Chris Evans', role='Captain America'), Actor(name='Mark Ruffalo', role='Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Black Widow'), Actor(name='Jeremy Renner', role='Hawkeye'), Actor(name='Don Cheadle', role='War Machine'), Actor(name='Paul Rudd', role='Ant-Man'), Actor(name='Brie Larson', role='Captain Marvel'), Actor(name='Karen Gillan', role='Nebula'), Actor(name='Danai Gurira', role='Okoye'), Actor(name='Bradley Cooper', role='Rocket'), Actor(name='Gwyneth Paltrow', role='Pepper Potts'), Actor(name='Josh Brolin', role='Thanos')], genres=['Science Fiction', 'Action', 'Adventure', 'Superhero'], budget=3500.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The Title of the movie"]
    year: Annotated[str, ..., "The year of the movie came"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[str, ..., "The rating of the movie out of skip, timepass, go for it, perfection"]

model_with_typeDict = model.with_structured_output(MovieDict)
responce = model_with_typeDict.invoke("provide the details about the movie Avengers 2")
responce
    

{'director': 'Joss Whedon',
 'rating': 'go for it',
 'title': 'Avengers: Age of Ultron',
 'year': '2015'}

In [24]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDataT(TypedDict):
    title:str
    cast:list[Actor]
    genres:list[str]
    rating:str


model_with_struc_3 = model.with_structured_output(MovieDataT)

res2 = model_with_struc_3.invoke("rovide the details about the movie Man of Steel")
res2

{'cast': [{'name': 'Henry Cavill', 'role': 'Clark Kent/Superman'},
  {'name': 'Amy Adams', 'role': 'Lois Lane'},
  {'name': 'Michael Shannon', 'role': 'General Zod'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'rating': '7.3/10',
 'title': 'Man of Steel'}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [25]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [26]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2b3cc4aa-f16a-4ea1-b7e8-9a045b985072'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 228, 'prompt_tokens': 204, 'total_tokens': 432, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dzih65tFbH4zfIQFPiny1bD4GVkw2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f46f9-7844-7d41-884d-5660a985a679-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output

In [27]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [28]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Hetvi Ahalpara, hetvi@constantsys.com, (123) 456-4077"}]
})

result["structured_response"]

{'name': 'Hetvi Ahalpara',
 'email': 'hetvi@constantsys.com',
 'phone': '(123) 456-4077'}

In [35]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class EmployInfo:
    """Employ info of Company"""
    name:str
    email:str
    phone:str
    salary:str
    position:str

agent2 = create_agent(
    model='gpt-5',
    response_format=EmployInfo
)

result2 = agent2.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: Hetvi Ahalpara, hetvi@constantsys.com, (123) 456-4077, 20000, Software Developer"}]
})

result2["structured_response"]

EmployInfo(name='Hetvi Ahalpara', email='hetvi@constantsys.com', phone='(123) 456-4077', salary='20000', position='Software Developer')